
7. Extend the GitHub Actions workflow to deploy to prod on merge to main using OIDC authentication (no stored secrets), including the correct permissions: id-token: write block. 

-->>

I tried to configure the GitHub Actions OIDC authentication for deploying the Databricks bundle to the `prod` environment. However, I could not complete the setup in Databricks Free Edition because the required account-level **Service Principal Federation Policy** is not available in the Free Edition.

The expected flow for the production setup is:

```text
Developer merges PR to main
          ↓
GitHub Actions starts
          ↓
id-token: write permission
          ↓
GitHub generates OIDC token
          ↓
Databricks verifies the token using Federation Policy
          ↓
Service Principal is authenticated
          ↓
databricks bundle deploy -t prod
          ↓
Deployment completed
```

The GitHub Actions workflow would contain:

```yaml
permissions:
  contents: read
  id-token: write
```

The `id-token: write` permission allows GitHub Actions to generate an OIDC token. Databricks then verifies that token against the Federation Policy, which can restrict access to a particular repository and branch.

For example, the policy can allow only:

```text
repo:https://github.com/rahul-dev010/Dabs_bakehouse/tree/main/Bakehouse_analytics
```

After successful authentication, the Databricks CLI can run:

```bash
databricks bundle deploy -t prod
```

This approach does not require storing a long-lived Databricks secret in GitHub Actions.

**Note:** I could not test the complete OIDC deployment flow because I am using Databricks Free Edition, which does not provide the required account-level federation policy configuration.






8. Design a rollback plan: if a bundle deploy to prod breaks a job, what CLI commands would you run to redeploy the previous working version quickly? 

-->>


**8. Rollback Plan

First, I created a `feature_1` branch to develop the feature, then pushed the changes and merged them into `main` through the pipeline. When a deployment issue occurs in production, my first step is to identify the last working commit using Git history.

```bash
git log --oneline
```

To quickly access the previous working version, I can check out its commit:

```bash
git switch --detach <previous-working-commit-hash>
```

This brings the previous version of the code into my local working directory. I can then redeploy that version to the production workspace using the Databricks CLI:

```bash
databricks bundle deploy -t prod
```

After confirming that the job is working correctly, I would fix the issue in the branch and use `git revert <faulty-commit-hash>` if I need to undo the faulty changes in the shared branch history. Finally, I would push the rollback commit and deploy the corrected version through the pipeline.

**Note:** Checking out an older commit does not automatically roll back production. The bundle must be deployed, and the production deployment target must be configured correctly.



9. Write a one-page onboarding guide for a new team member explaining how a change moves from a local databricks.yml edit to running safely in production, referencing the CLI, the bundle lifecycle, and the CI/CD workflow together. 

-->>

# Onboarding Guide: From Local Edit to Production

Welcome to the team! Here's how a change actually gets from your laptop to running safely in production.

## 1. Local Setup (CLI + U2M)
Install the Databricks CLI and log in to the **dev** workspace using **U2M** (`databricks auth login`). This opens a browser, you log in as yourself, and the CLI stores a short-lived token. This is only for personal dev work.

## 2. Edit the Bundle
Our jobs live as code, not UI clicks — in `databricks.yml` and `resources/*.yml` (a **Databricks Asset Bundle**). Make your change here.

## 3. Test Locally

``` 
databricks bundle validate -t dev
databricks bundle deploy -t dev
databricks bundle run my_job -t dev
```

`validate` catches YAML mistakes, `deploy` creates/updates the job in dev, `run` lets you confirm it actually works — all safely under your own login.

## 4. Git + Pull Request
Push your branch and open a PR into `main`. A teammate reviews it before anything moves closer to prod.

## 5. Merge Triggers CI/CD
Merging to `main` kicks off a **GitHub Actions** workflow that runs the same `validate` → `deploy`, but targeting `prod`.

## 6. Authentication (No Human, No Stored Secret)
Prod deploys don't use a personal login or even a stored password. GitHub generates a short-lived **OIDC** token per run, and Databricks trusts it via a **Federation Policy** tied to our repo/branch. The workflow needs `permissions: id-token: write` for this to work.

## 7. Job Runs as a Service Principal
In `databricks.yml`, the `prod` target sets `run_as: service_principal_name`. So the job is owned by a robot account, not a person — no breakage if someone leaves the team.

## 8. Rollback if Needed
We tag every successful prod deploy. To roll back: `git checkout <last-good-tag>` → `bundle validate -t prod` → `bundle deploy -t prod`.

## TL;DR
Edit locally (U2M) → test in dev → PR review → merge → GitHub Actions deploys via OIDC → job runs as a Service Principal in prod → Git tags make rollback easy.